In [ ]:
import os
os.environ['PYSPARK_SUBMIT_ARGS'] = \
    '--packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.8 pyspark-shell'

In [ ]:
import findspark
findspark.init("/home/asf/bigdata/spark")

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from influxdb_client import InfluxDBClient, Point
from influxdb_client.client.write_api import SYNCHRONOUS

spark = SparkSession.builder \
    .appName("Test") \
    .master("local[*]") \
    .config("spark.sql.streaming.forceDeleteTempCheckpointLocation", "true") \
    .getOrCreate()

spark
print("Spark Session Created Successfully with Kafka Support!")

Spark Session Created Successfully with Kafka Support!


In [ ]:
from pyspark.sql.functions import col, from_json, to_timestamp
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from influxdb_client import InfluxDBClient, Point
from influxdb_client.client.write_api import SYNCHRONOUS

# Set logging level to avoid spamming the console
spark.sparkContext.setLogLevel("WARN")

# 2. Connect to Kafka and read the "btcusd" stream
# Kafka sources emit binary data by default (key and value)
kafka_raw_df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribe", "btcusd") \
    .option("startingOffsets", "latest") \
    .load()

# 3. Cast the binary payload to a readable String
# Typically, crypto tickers send data as JSON
kafka_string_df = kafka_raw_df.selectExpr("CAST(value AS STRING) as json_payload")

# 4. Define the schema matching your "btcusd" JSON payloads
# (Modify this schema to perfectly fit your incoming WebSocket/API structure)
btcusd_schema = StructType([
    StructField("symbol", StringType(), True),
    StructField("bid", DoubleType(), True),
    StructField("ask", DoubleType(), True),
    StructField("last", DoubleType(), True),
    StructField("volume", DoubleType(), True),
    StructField("timestamp", StringType(), True), # Read as string first to parse ISO format
    StructField("source", StringType(), True),
    StructField("high", DoubleType(), True),
    StructField("low", DoubleType(), True),
    StructField("direction", StringType(), True),
    StructField("dayDiffPercent", DoubleType(), True),
    StructField("description", StringType(), True),
    StructField("time", StringType(), True),
    StructField("spread", DoubleType(), True),
    StructField("mid", DoubleType(), True)
])

# 2. InfluxDB Configuration
INFLUX_URL = "http://localhost:8086"
INFLUX_TOKEN = "KPQNZeSch1QKAThcn8BM4WO2_THzdaj29dql9MX054lmUSLnJSGc70MTzuPthvTOVmeIVixZItwBEvECbpO3Xg=="
INFLUX_ORG = "admin"
INFLUX_BUCKET = "btcusd"

# 5. Parse the JSON string into structured columns
parsed_df = kafka_string_df \
    .withColumn("data", from_json(col("json_payload"), btcusd_schema)) \
    .select("data.*") \
    .withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd'T'HH:mm:ss'Z'"))


# 6. Micro-batch writer function to process and send metrics to InfluxDB
def write_to_influx(batch_df, batch_id):
    if batch_df.isEmpty():
        return

    # Open connection inside the worker context
    with InfluxDBClient(url=INFLUX_URL, token=INFLUX_TOKEN, org=INFLUX_ORG) as client:
        write_api = client.write_api(write_options=SYNCHRONOUS)
        rows = batch_df.collect()
        points = []

        for row in rows:
            # Safe fallbacks if any fields return null
            crypto_symbol = row["symbol"] if row["symbol"] else "BTCUSD"
            data_source = row["source"] if row["source"] else "unknown"

            # Map metric fields into a flat time-series point
            point = Point("bitcoin_ticker_metrics") \
                .tag("crypto_symbol", crypto_symbol) \
                .tag("source", data_source) \
                .field("bid", float(row["bid"]) if row["bid"] is not None else 0.0) \
                .field("ask", float(row["ask"]) if row["ask"] is not None else 0.0) \
                .field("last_price", float(row["last"]) if row["last"] is not None else 0.0) \
                .field("volume", float(row["volume"]) if row["volume"] is not None else 0.0) \
                .field("high", float(row["high"]) if row["high"] is not None else 0.0) \
                .field("low", float(row["low"]) if row["low"] is not None else 0.0) \
                .field("spread", float(row["spread"]) if row["spread"] is not None else 0.0) \
                .field("mid", float(row["mid"]) if row["mid"] is not None else 0.0) \
                .field("dayDiffPercent", float(row["dayDiffPercent"]) if row["dayDiffPercent"] is not None else 0.0)

            # Explicitly bind the parsed log timestamp if available
            if row["timestamp"]:
                point.time(row["timestamp"])

            points.append(point)

        if points:
            write_api.write(bucket=INFLUX_BUCKET, org=INFLUX_ORG, record=points)


# 7. Start running the stream in the background
query = parsed_df.writeStream \
    .foreachBatch(write_to_influx) \
    .option("checkpointLocation", "/tmp/spark_kafka_btcusd_checkpoint") \
    .start()

print("Streaming query successfully started and running in the background...")

26/06/11 23:39:28 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/06/11 23:39:28 WARN StreamingQueryManager: Stopping existing streaming query [id=6149655c-b8e6-4faf-864a-b59e7f072f11, runId=2a799305-d270-4dcd-bb24-71dc39fb1980], as a new run is being started.


Streaming query successfully started and running in the background...


26/06/11 23:39:28 WARN TaskSetManager: Lost task 0.0 in stage 1857.0 (TID 2089) (192.168.240.128 executor driver): TaskKilled (Stage cancelled: Job 1854 cancelled part of cancelled job group 2a799305-d270-4dcd-bb24-71dc39fb1980)
26/06/11 23:39:30 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
                                                                                